# stage 3 — `%%ggb <host>`: the Construction type's Python surface

The cell body is parsed by the closed-world parser (28 heads, `:label` refs), planned into ONE `Eval` and applied to the host named on the magic line (no hidden applet: v1 looked for `GeoGebra._instance` / `user_ns['ggb']`). The check is the stage 2 gate #2 group g1 (lesson 04, 10 statements, class `exact`): the `<construction>` the applet holds after the magic must be byte-identical to `probes/stage2/gate2/v2_g1.xml`, which the same host produced from the rendered strings on 2026-09-07.

In [ ]:
import sys, json, hashlib, time; sys.path.insert(0, '/Users/manabu/work/ggblab-replay')
import ggblab.host.html_host as H; H.DEPLOY = 'https://cdn.geogebra.org/apps/deployggb.js'
from ggblab import GeoGebra
from ggblab.xml_errata import construction_xml
%load_ext ggblab.ipymagic
G1 = json.load(open('/Users/manabu/work/ggblab-replay/probes/stage2/gate2/groups.json'))[1]
REF = open('/Users/manabu/work/ggblab-replay/probes/stage2/gate2/v2_g1.xml', encoding='utf-8').read()
md5 = lambda s: hashlib.md5(s.encode('utf-8')).hexdigest()[:8]
print('g1 bodies', len(G1['bodies']), '| ref <construction> md5', md5(construction_xml(REF)))

In [ ]:
g = GeoGebra(appName='suite', showToolBar=True, showZoomButtons=True, showAlgebraInput=True, showMenuBar=True, algebraInputPosition='top'); g

## 1. the plan (nothing is sent): the 10 bodies → one `Eval` whose strings are gate #1's `v2` strings

In [ ]:
%%ggb g --plan
A=(2, 1)
B=(-1, 2)
C=(0, -1)
Polygon(:A, :B, :C)
l_BC = PerpendicularBisector(:B, :C)
l_CA = PerpendicularBisector(:C, :A)
l_AB = PerpendicularBisector(:A, :B)
O = Intersect(:l_BC, :l_CA)
Circle(:O, :A)
TriangleCenter(:A, :B, :C, 3)

In [ ]:
print('plan == gate v2 strings:', list(_[0].commands) == G1['v2'])

## 2. send it (first call in a fresh page: `TriangleCenter` is known to fail lazily — gate #2 副産物 ①), then `:const :new` and send again

In [ ]:
%%ggb g
A=(2, 1)
B=(-1, 2)
C=(0, -1)
Polygon(:A, :B, :C)
l_BC = PerpendicularBisector(:B, :C)
l_CA = PerpendicularBisector(:C, :A)
l_AB = PerpendicularBisector(:A, :B)
O = Intersect(:l_BC, :l_CA)
Circle(:O, :A)
TriangleCenter(:A, :B, :C, 3)

In [ ]:
first = _; print('first labels', first)   # a {'error': …} entry = the Apps API threw for that command (TriangleCenter loads its module lazily); the batch kept going
import time; time.sleep(3)   # let the discrete-commands module finish loading before the second send

In [ ]:
%%ggb g
:const :new
A=(2, 1)
B=(-1, 2)
C=(0, -1)
Polygon(:A, :B, :C)
l_BC = PerpendicularBisector(:B, :C)
l_CA = PerpendicularBisector(:C, :A)
l_AB = PerpendicularBisector(:A, :B)
O = Intersect(:l_BC, :l_CA)
Circle(:O, :A)
TriangleCenter(:A, :B, :C, 3)

In [ ]:
second = _; print('second labels', second)
x = g.xml(timeout=60); c = construction_xml(x)
print('LABELS', len([l for l in second if isinstance(l, str)]), '| <construction> md5', md5(c), '| BYTE_EQUAL_TO_GATE2_G1', c == construction_xml(REF))
print('errors pulled:', g.errors())

In [ ]:
print('DONE')